# dim_indicators

Builds `pondolens.dev_gold.dim_indicator`, the indicator dimension of the galaxy schema. `FACT_REGIONAL_INDICATOR` looks up `indicator_id` here.

**Sources (Silver)**
| indicator | Silver table | measure_type |
|---|---|---|
| `unemployment` | `pondolens.dev_silver.unemployment_rate` | `unemployment_rate` |
| `grdp` | `pondolens.dev_silver.grdp_per_capita` | `grdp_per_capita` (one row per `price_basis`) |
| `service_access` | `pondolens.dev_silver.service_access` | from `service_measure` (water, electricity, internet) |

**Columns:** `indicator_id | indicator | measure_type | price_basis | unit | service | breakdown_type | breakdown_value`

**Rules**
- **Business key:** `indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value`. One row per unique combination. The MERGE matches on this key.
- `indicator_id`: team range **40000** → first ID is **40001**. Existing indicators keep their ID; only new indicators get `MAX(indicator_id) + ROW_NUMBER()`, applied to new indicators only. IDs must stay below 50000 (FACT_INDICATORS range).
- **Not applicable** attributes are stored as `'N/A'` instead of NULL, so dashboard filters look clean and fact lookups can use plain `=` joins.
- `price_basis` (GRDP only): `'2018'` = constant 2018 prices; `'2021'`–`'2025'` = current prices of that year. Others = `'N/A'`.
- Unemployment and GRDP are regional totals, so `breakdown_type = 'Total'` and `breakdown_value = 'All'`.
- `unit`: anything with `%` or "percent" becomes `'percent'`; GRDP per capita is `'pesos'`.
- **Retired indicators are kept, never deleted.** Facts may still point to them. Step 7 lists them for review.
- Reusable and idempotent: rerunning with the same Silver data inserts nothing.

**Safe if two people run it at the same time (run lock)**
- Step 2 claims a lock row for `dim_indicator` in `pondolens.dev_gold.etl_run_lock`. Only one run can hold it.
- If another run already holds the lock, this run **stops before writing anything** with the message *"Another run is in progress"*. Wait a minute and run it again.
- If two runs try to claim the lock at the exact same moment, Delta lets only one save the claim; the other fails with a *concurrent update* error. That is expected and safe: nothing was written, just run it again.
- The lock is released at the end (step 10). If a run fails halfway, the lock expires by itself after 10 minutes, or you can release it with the cell at the bottom.

## 1. Create the Gold table

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pondolens.dev_gold;

CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_indicator (
  indicator_id INT NOT NULL COMMENT 'Surrogate key. Range 40001-49999.',
  indicator STRING NOT NULL COMMENT 'unemployment, grdp, service_access',
  measure_type STRING NOT NULL COMMENT 'unemployment_rate, grdp_per_capita, electricity_access, ...',
  price_basis STRING NOT NULL COMMENT '2018 = constant 2018 prices; 2021-2025 = current prices; N/A otherwise',
  unit STRING NOT NULL COMMENT 'percent, pesos, ...',
  service STRING NOT NULL COMMENT 'electricity, water, internet; N/A otherwise',
  breakdown_type STRING NOT NULL COMMENT 'Total, Residence, ...',
  breakdown_value STRING NOT NULL COMMENT 'All, All People, All families, ...',
  CONSTRAINT dim_indicator_pk PRIMARY KEY (indicator_id)
)
USING DELTA
COMMENT 'Gold dimension: one row per indicator measure. Business key: indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value.';

-- Shared lock table: one row per target table. Teammates can reuse it for their own tables.
CREATE TABLE IF NOT EXISTS pondolens.dev_gold.etl_run_lock (
  table_name STRING NOT NULL,
  locked_by STRING,
  locked_at TIMESTAMP
)
USING DELTA
COMMENT 'Run lock: prevents two runs from writing to the same table at the same time.';

-- Add the lock row for this table once (does nothing if it already exists).
MERGE INTO pondolens.dev_gold.etl_run_lock AS l
USING (SELECT 'dim_indicator' AS table_name) AS s
ON l.table_name = s.table_name
WHEN NOT MATCHED THEN INSERT (table_name, locked_by, locked_at)
VALUES (s.table_name, NULL, NULL);

## 2. Claim the run lock

Claims the lock only if nobody holds it (or the last holder's lock is older than 10 minutes, meaning that run failed).

- **Stops with "Another run is in progress"** → someone else is running this notebook. Wait for it to finish, then run again.
- **Fails with a "concurrent update" / `ConcurrentModificationException` error** → two runs claimed at the same instant and the other one won. Nothing was written; run again.

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_run_id STRING;
SET VAR v_run_id = REPLACE(UUID(), '-', '');

-- Both runs update the SAME row, so Delta only lets one of them commit.
UPDATE pondolens.dev_gold.etl_run_lock
SET locked_by = v_run_id,
    locked_at = CURRENT_TIMESTAMP()
WHERE table_name = 'dim_indicator'
  AND (locked_by IS NULL OR locked_at < CURRENT_TIMESTAMP() - INTERVAL 10 MINUTES);

-- Continue only if this run holds the lock.
SELECT ASSERT_TRUE(
  COUNT(*) > 0 AND COUNT_IF(locked_by = v_run_id) = COUNT(*),
  'Another run is in progress on dim_indicator. Wait for it to finish, then run again.'
)
FROM pondolens.dev_gold.etl_run_lock
WHERE table_name = 'dim_indicator';

## 3. Build the indicator list from Silver

Each Silver table is reduced to its distinct indicator combinations first (`SELECT DISTINCT` per source), so only a few rows reach the join and the MERGE.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_indicator_stage AS
WITH unemployment AS (
  SELECT DISTINCT
    'unemployment' AS indicator,
    'unemployment_rate' AS measure_type,
    'N/A' AS price_basis,
    'percent' AS unit,
    'N/A' AS service,
    'Total' AS breakdown_type,
    'All' AS breakdown_value
  FROM pondolens.dev_silver.unemployment_rate
),
grdp AS (
  SELECT DISTINCT
    'grdp' AS indicator,
    'grdp_per_capita' AS measure_type,
    CAST(price_basis AS STRING) AS price_basis,
    'pesos' AS unit,
    'N/A' AS service,
    'Total' AS breakdown_type,
    'All' AS breakdown_value
  FROM pondolens.dev_silver.grdp_per_capita
),
service_access AS (
  SELECT DISTINCT
    'service_access' AS indicator,
    -- 'Electricity access' -> 'electricity_access'
    LOWER(TRIM(BOTH '_' FROM REGEXP_REPLACE(TRIM(service_measure), '[^A-Za-z0-9]+', '_'))) AS measure_type,
    'N/A' AS price_basis,
    CASE
      WHEN INSTR(unit, '%') > 0 OR LOWER(unit) LIKE '%percent%' THEN 'percent'
      ELSE LOWER(TRIM(unit))
    END AS unit,
    COALESCE(NULLIF(LOWER(TRIM(service)), ''), 'N/A') AS service,
    COALESCE(NULLIF(TRIM(breakdown_type), ''), 'N/A') AS breakdown_type,
    COALESCE(NULLIF(TRIM(breakdown_value), ''), 'N/A') AS breakdown_value
  FROM pondolens.dev_silver.service_access
)
SELECT * FROM unemployment
UNION
SELECT * FROM grdp
UNION
SELECT * FROM service_access;

SELECT * FROM dim_indicator_stage ORDER BY indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value;

## 4. Data quality checks on the staged indicators

In [0]:
%sql
SELECT
  ASSERT_TRUE(COUNT(*) > 0,
    'No indicators found in Silver.') AS has_rows,
  ASSERT_TRUE(
    COUNT_IF(
      NULLIF(TRIM(measure_type), '') IS NULL
      OR NULLIF(TRIM(price_basis), '') IS NULL
      OR NULLIF(TRIM(unit), '') IS NULL
    ) = 0,
    'Blank measure_type, price_basis or unit. Check the Silver source.') AS keys_present,
  ASSERT_TRUE(COUNT_IF(indicator = 'grdp' AND price_basis NOT RLIKE '^[0-9]{4}$') = 0,
    'GRDP price_basis must be a 4-digit year.') AS valid_price_basis
FROM dim_indicator_stage;

## 5. Check the existing Gold table

In [0]:
%sql
SELECT
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT indicator_id),
    'Duplicate indicator_id in Gold.') AS unique_ids,
  ASSERT_TRUE(COUNT_IF(indicator_id NOT BETWEEN 40001 AND 49999) = 0,
    'indicator_id outside the 40001-49999 range.') AS ids_in_range
FROM pondolens.dev_gold.dim_indicator;

-- Duplicate business keys (should return no rows).
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate business keys in Gold.')
FROM (
  SELECT indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value
  FROM pondolens.dev_gold.dim_indicator
  GROUP BY indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value
  HAVING COUNT(*) > 1
);

## 6. Assign indicator_id to new indicators only

- An indicator is *new* when its business key is not yet in Gold.
- `ROW_NUMBER()` runs on new indicators only and continues from the last allocated ID (40000 when the table is empty).
- Existing indicators are not renumbered, so their IDs never change.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_indicator_new_ids AS
WITH new_indicators AS (
  SELECT s.*
  FROM dim_indicator_stage AS s
  LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS t
    ON s.indicator = t.indicator
    AND s.measure_type = t.measure_type
    AND s.price_basis = t.price_basis
    AND s.unit = t.unit
    AND s.service = t.service
    AND s.breakdown_type = t.breakdown_type
    AND s.breakdown_value = t.breakdown_value
),
last_id AS (
  -- Last allocated ID; 40000 when the table is empty.
  SELECT COALESCE(MAX(indicator_id), 40000) AS last_number
  FROM pondolens.dev_gold.dim_indicator
)
SELECT
  n.*,
  l.last_number + ROW_NUMBER() OVER (ORDER BY n.indicator, n.measure_type, n.price_basis, n.unit, n.service, n.breakdown_type, n.breakdown_value) AS indicator_id
FROM new_indicators AS n
CROSS JOIN last_id AS l;

-- New IDs must stay inside the DIM_INDICATOR range.
SELECT ASSERT_TRUE(
  COUNT_IF(indicator_id NOT BETWEEN 40001 AND 49999) = 0,
  'New indicator_id values must be between 40001 and 49999.'
)
FROM dim_indicator_new_ids;

-- Indicators that will be added in this run.
SELECT * FROM dim_indicator_new_ids ORDER BY indicator_id;

## 7. Retired indicators (review only)

Indicators in Gold that no longer appear in any Silver table. They are **kept**, because fact rows may still reference their `indicator_id`. This list is empty on a normal run.

In [0]:
%sql
SELECT t.*
FROM pondolens.dev_gold.dim_indicator AS t
LEFT ANTI JOIN dim_indicator_stage AS s
  ON t.indicator = s.indicator
    AND t.measure_type = s.measure_type
    AND t.price_basis = s.price_basis
    AND t.unit = s.unit
    AND t.service = s.service
    AND t.breakdown_type = s.breakdown_type
    AND t.breakdown_value = s.breakdown_value
ORDER BY t.indicator_id;

## 8. Merge into Gold using the business key

Every column except `indicator_id` is part of the business key, so a matched row has nothing to update. The MERGE only inserts new indicators. Rerunning with unchanged Silver data writes 0 rows.

The first statement re-checks the lock right before writing, in case this run took longer than the lock expiry.

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) > 0 AND COUNT_IF(locked_by = v_run_id) = COUNT(*),
  'This run no longer holds the lock (it expired or was released). Nothing was written; run the notebook again.'
)
FROM pondolens.dev_gold.etl_run_lock
WHERE table_name = 'dim_indicator';

MERGE INTO pondolens.dev_gold.dim_indicator AS t
USING dim_indicator_new_ids AS s
ON t.indicator = s.indicator
    AND t.measure_type = s.measure_type
    AND t.price_basis = s.price_basis
    AND t.unit = s.unit
    AND t.service = s.service
    AND t.breakdown_type = s.breakdown_type
    AND t.breakdown_value = s.breakdown_value

WHEN NOT MATCHED THEN INSERT (
  indicator_id,
  indicator,
  measure_type,
  price_basis,
  unit,
  service,
  breakdown_type,
  breakdown_value
)
VALUES (
  CAST(s.indicator_id AS INT),
  s.indicator,
  s.measure_type,
  s.price_basis,
  s.unit,
  s.service,
  s.breakdown_type,
  s.breakdown_value
);

## 9. Post-merge checks and summary

Final safety net: confirms IDs and business keys are still unique after the write.

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT indicator_id)
  AND COUNT(*) = COUNT(DISTINCT STRUCT(indicator, measure_type, price_basis, unit, service, breakdown_type, breakdown_value)),
  'Duplicate IDs or business keys after the merge. Check for a concurrent run.'
)
FROM pondolens.dev_gold.dim_indicator;

-- Every Silver indicator must now exist in Gold.
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Some Silver indicators are missing from Gold.')
FROM dim_indicator_stage AS s
LEFT ANTI JOIN pondolens.dev_gold.dim_indicator AS t
  ON s.indicator = t.indicator
    AND s.measure_type = t.measure_type
    AND s.price_basis = t.price_basis
    AND s.unit = t.unit
    AND s.service = t.service
    AND s.breakdown_type = t.breakdown_type
    AND s.breakdown_value = t.breakdown_value;

SELECT
  indicator,
  COUNT(*) AS indicator_rows,
  MIN(indicator_id) AS min_id,
  MAX(indicator_id) AS max_id
FROM pondolens.dev_gold.dim_indicator
GROUP BY indicator
ORDER BY min_id;

SELECT * FROM pondolens.dev_gold.dim_indicator ORDER BY indicator_id;

## 10. Release the run lock

In [0]:
%sql
UPDATE pondolens.dev_gold.etl_run_lock
SET locked_by = NULL,
    locked_at = NULL
WHERE table_name = 'dim_indicator'
  AND locked_by = v_run_id;

## Manual unlock (only if a run failed halfway)

If a run stopped with an error, its lock stays for up to 10 minutes. To run again sooner, **first make sure nobody else is running this notebook**, then run only this cell.

In [0]:
%sql
-- UPDATE pondolens.dev_gold.etl_run_lock
-- SET locked_by = NULL, locked_at = NULL
-- WHERE table_name = 'dim_indicator';

## For FACT_REGIONAL_INDICATOR (lookup example)

How the fact notebook can find `indicator_id` for each Silver row. Plain `=` joins work because there are no NULLs in the key.

```sql
-- Unemployment
SELECT d.indicator_id, u.year, u.region_code, u.unemployment_rate AS measurement
FROM pondolens.dev_silver.unemployment_rate AS u
JOIN pondolens.dev_gold.dim_indicator AS d
  ON d.indicator = 'unemployment' AND d.measure_type = 'unemployment_rate';

-- GRDP per capita (one indicator per price_basis)
SELECT d.indicator_id, g.year, g.region_code, g.grdp_per_capita AS measurement
FROM pondolens.dev_silver.grdp_per_capita AS g
JOIN pondolens.dev_gold.dim_indicator AS d
  ON d.indicator = 'grdp' AND d.price_basis = CAST(g.price_basis AS STRING);
```

For service access, apply the same `measure_type`, `unit`, `service` and `breakdown` cleaning as in step 3, then join on all seven key columns.